# NeuroGPT + class-conditional soft prompt-tuning (our novelty)

> Apply soft/continuous prompt-tuning — learn a small set of virtual prompt tokens prepended to the frozen EEG sequence, one learned set per mental-state condition, steering the frozen model's internal representations.

This notebook clones the plain vendored repo and writes our added `src/prompt_tuning/` module and smoke test into it (embedded below, verbatim from the local implementation), so it's fully self-contained. See `PROMPT_TUNING.md` in the main repo for the full design rationale — in short: a fixed, label-independent "menu" of per-condition soft-prompt tokens is prepended to *every* input, and the entire backbone is frozen except the prompt menu + the existing pooler/decoding head.
**Before running**: Runtime > Change runtime type > GPU (T4 is enough).

This notebook clones the original vendored repo fresh — it does not depend on any local files, so it runs standalone in Colab.

## 1. Environment

In [ ]:
!nvidia-smi

In [ ]:
!git clone https://github.com/wenhui0206/NeuroGPT.git
%cd NeuroGPT
!pip install -q -r requirements.txt

## 2. Add the soft prompt-tuning module (no vendored files touched)

In [ ]:
!mkdir -p src/prompt_tuning tests

In [ ]:
%%writefile src/prompt_tuning/__init__.py


In [ ]:
%%writefile src/prompt_tuning/soft_prompt.py
#!/usr/bin/env python3
"""
Class-conditional soft prompt-tuning for NeuroGPT.

Novelty (see PROMPT_TUNING.md): learn a small, fixed "menu" of virtual
prompt tokens, one contiguous block per mental-state condition, and prepend
the *entire* menu to every input sequence (train and inference alike). The
frozen backbone must learn to attend to the block relevant to the trial's
actual EEG content -- no label is used to select a prompt, so nothing about
the answer leaks into the input. Freeze everything except the prompt menu
and the existing pooler/decoding head.
"""
import os
import sys
from typing import Dict

import torch
import torch.nn as nn

_SRC_DIR = os.path.dirname(os.path.dirname(os.path.realpath(__file__)))
if _SRC_DIR not in sys.path:
    sys.path.insert(0, _SRC_DIR)

from model import Model  # noqa: E402  (vendored NeuroGPT model)


class ConditionSoftPrompt(nn.Module):
    """
    Holds a (num_conditions, num_tokens_per_condition, embed_dim) parameter
    and prepends the flattened, batch-broadcast menu to inputs_embeds /
    attention_mask. num_tokens_per_condition=0 is a no-op (frozen linear
    probe baseline).
    """

    def __init__(self, num_conditions: int, num_tokens_per_condition: int, embed_dim: int) -> None:
        super().__init__()
        self.num_conditions = num_conditions
        self.num_tokens_per_condition = num_tokens_per_condition
        self.embed_dim = embed_dim
        self.num_prompt_tokens = num_conditions * num_tokens_per_condition

        if self.num_prompt_tokens > 0:
            self.prompt_embeds = nn.Parameter(
                torch.randn(num_conditions, num_tokens_per_condition, embed_dim) * 0.02
            )
        else:
            self.prompt_embeds = None

    def forward(self, inputs_embeds: torch.Tensor, attention_mask: torch.Tensor):
        if self.prompt_embeds is None:
            return inputs_embeds, attention_mask

        batch_size = inputs_embeds.size(0)
        menu = self.prompt_embeds.reshape(1, self.num_prompt_tokens, self.embed_dim)
        menu = menu.expand(batch_size, -1, -1).to(dtype=inputs_embeds.dtype)

        prompted_embeds = torch.cat([menu, inputs_embeds], dim=1)

        prompt_mask = torch.ones(
            batch_size, self.num_prompt_tokens,
            dtype=attention_mask.dtype, device=attention_mask.device,
        )
        prompted_mask = torch.cat([prompt_mask, attention_mask], dim=1)

        return prompted_embeds, prompted_mask


class PromptTunedModel(Model):
    """
    Wraps the vendored Model with a ConditionSoftPrompt. forward() is a full
    method override (not a monkeypatch or super()-partial-reuse) since
    Model.forward is a single monolithic method not designed for partial
    reuse -- this is the standard "extend via subclass" pattern for
    vendored code we don't own.

    Decoding-mode only: the reconstruction loss used in CSM pretraining
    assumes the sequence length is unmodified, so this wrapper refuses to
    prepend prompts outside decoding mode.
    """

    def __init__(
        self,
        encoder,
        embedder,
        decoder,
        unembedder=None,
        num_conditions: int = 4,
        num_tokens_per_condition: int = 4,
    ) -> None:
        super().__init__(encoder=encoder, embedder=embedder, decoder=decoder, unembedder=unembedder)
        self.soft_prompt = ConditionSoftPrompt(
            num_conditions=num_conditions,
            num_tokens_per_condition=num_tokens_per_condition,
            embed_dim=decoder.embed_dim,
        )

    def forward(
        self,
        batch: Dict[str, torch.Tensor],
        prep_batch: bool = True,
        return_batch: bool = False,
    ):
        if self.encoder is not None:
            features = self.encoder(batch['inputs'])
            if self.is_decoding_mode and self.ft_only_encoder:
                outputs = {'outputs': features, 'decoding_logits': features}
                return (outputs, batch) if return_batch else outputs

            b, f1, f2 = features.size()
            nchunks = batch['inputs'].size()[1]
            batch['inputs'] = features.view(b // nchunks, nchunks, f1 * f2)

        if prep_batch:
            if len(batch['inputs'].size()) > 3:
                bsize, chunk, chann, time = batch['inputs'].size()
                batch['inputs'] = batch['inputs'].view(bsize, chunk, chann * time)
            batch = self.prep_batch(batch=batch)
        else:
            assert 'inputs_embeds' in batch, 'inputs_embeds not in batch'

        batch['inputs_embeds'] = self.embedder(batch=batch)

        if not self.is_decoding_mode and self.soft_prompt.num_prompt_tokens > 0:
            raise RuntimeError(
                "PromptTunedModel only supports prepending prompt tokens in "
                "decoding mode (is_decoding_mode=True); the CSM pretraining "
                "reconstruction loss assumes the sequence length equals the "
                "original number of chunks, which prompt-prepending breaks."
            )

        batch['inputs_embeds'], batch['attention_mask'] = self.soft_prompt(
            inputs_embeds=batch['inputs_embeds'],
            attention_mask=batch['attention_mask'],
        )
        # Defensive: keep attention_mask an integer dtype for the pooler's
        # advanced indexing (attention_mask.sum(dim=1)-1), regardless of
        # what dtype BaseEmbedder.prep_batch cast it to upstream.
        batch['attention_mask'] = batch['attention_mask'].long()

        outputs = self.decoder(batch=batch)

        if self.unembedder is not None and not self.is_decoding_mode:
            outputs['outputs'] = self.unembedder(inputs=outputs['outputs'])['outputs']

        return (outputs, batch) if return_batch else outputs


def freeze_for_prompt_tuning(model: "PromptTunedModel") -> None:
    """
    Freezes encoder, embedder, and the decoder's transformer backbone,
    leaving only the soft-prompt menu and the existing pooler/decoding head
    trainable. Mirrors the granularity of train_gpt.py's --freeze-* flags,
    but is specific to this method so it lives here rather than editing
    vendored code.
    """
    if model.encoder is not None:
        for param in model.encoder.parameters():
            param.requires_grad = False

    for param in model.embedder.parameters():
        param.requires_grad = False

    for name, param in model.decoder.named_parameters():
        if 'pooler_layer' in name or 'decoding_head' in name:
            continue
        param.requires_grad = False

    if model.unembedder is not None:
        for param in model.unembedder.parameters():
            param.requires_grad = False

    # model.soft_prompt.prompt_embeds keeps requires_grad=True (nn.Parameter default)

In [ ]:
%%writefile src/prompt_tuning/train_prompt_tuning.py
#!/usr/bin/env python3
"""
Fine-tune NeuroGPT for downstream classification with the class-conditional
soft-prompt-tuning novelty (see PROMPT_TUNING.md), instead of the paper's
own fine-tuning approach (that's what train_gpt.py + scripts/finetune.sh
already do -- see PROMPT_TUNING.md for the "actual" baseline command).

This script is decoding-mode only. It reuses train_gpt.py's argument
parser, dataset construction, and HF Trainer plumbing (make_trainer)
unmodified, swapping in only the model constructor: instead of a plain
frozen/unfrozen Model, it builds a PromptTunedModel and applies
freeze_for_prompt_tuning.

BCI-IV-2a's 4 motor-imagery classes stand in here for "mental-state
conditions" -- no focused/mind-wandering EEG dataset ships with the
NeuroGPT repo. The mechanism itself is dataset-agnostic: swapping in a
real attention-state dataset only requires a different Dataset class and
--num-decoding-classes.

Leave the original --freeze-* flags at their defaults (False) when using
this script; --freeze-backbone (default True) does the freezing instead,
uniformly across encoder/embedder/decoder-transformer.
"""
import json
import os
import sys
from typing import Dict

import numpy as np
import pandas as pd
from numpy import random
from torch import manual_seed

_SRC_DIR = os.path.dirname(os.path.dirname(os.path.realpath(__file__)))
if _SRC_DIR not in sys.path:
    sys.path.insert(0, _SRC_DIR)

import train_gpt  # noqa: E402  (vendored NeuroGPT training entrypoint)
from prompt_tuning.soft_prompt import PromptTunedModel, freeze_for_prompt_tuning  # noqa: E402


def get_args():
    parser = train_gpt.get_args()
    parser.add_argument(
        '--num-prompt-tokens-per-condition',
        metavar='INT', default=4, type=int,
        help='number of learned virtual prompt tokens per mental-state '
             'condition; 0 disables prompting (frozen linear-probe baseline) '
             '(default: 4)'
    )
    parser.add_argument(
        '--freeze-backbone',
        metavar='BOOL', default='True', choices=('True', 'False'), type=str,
        help='whether to freeze encoder/embedder/decoder-transformer and '
             'train only the soft-prompt menu + pooler + decoding head '
             '(default: True). Leave the original --freeze-* flags at '
             'their defaults (False) when using this flag.'
    )
    return parser


def make_prompt_tuned_model(config: Dict):
    base_model = train_gpt.make_model(dict(config))

    model = PromptTunedModel(
        encoder=base_model.encoder,
        embedder=base_model.embedder,
        decoder=base_model.decoder,
        unembedder=base_model.unembedder,
        num_conditions=config['num_decoding_classes'],
        num_tokens_per_condition=config['num_prompt_tokens_per_condition'],
    )
    # base_model already ran switch_decoding_mode() inside make_model()
    # (same shared decoder/embedder objects), so just copy the top-level
    # flags rather than re-invoking switch_decoding_mode (which would
    # re-warn about the pooler/head "already existing").
    model.is_decoding_mode = base_model.is_decoding_mode
    model.ft_only_encoder = base_model.ft_only_encoder

    if config['freeze_backbone']:
        freeze_for_prompt_tuning(model)

    return model


def train(config: Dict = None):
    if config is None:
        config = train_gpt.get_config(get_args().parse_args())

    assert config['training_style'] == 'decoding', (
        "Soft prompt-tuning is implemented for the decoding/classification "
        "setting only (see PROMPT_TUNING.md); CSM pretraining is unaffected "
        "and should keep using train_gpt.py directly."
    )

    if config['do_train']:
        os.makedirs(config['log_dir'], exist_ok=True)
        config_filepath = os.path.join(config['log_dir'], 'train_config.json')
        with open(config_filepath, 'w') as f:
            json.dump(config, f, indent=2)
        config['resume_from'] = None

    if config['set_seed']:
        random.seed(config['seed'])
        manual_seed(config['seed'])

    downstream_path = config['dst_data_path']
    train_folds, test_folds = train_gpt.cv_split_bci(sorted(os.listdir(downstream_path))[:18])
    train_files = train_folds[config['fold_i']]
    test_files = test_folds[config['fold_i']]

    dataset_kwargs = dict(
        sample_keys=['inputs', 'attention_mask'],
        chunk_len=config['chunk_len'],
        num_chunks=config['num_chunks'],
        ovlp=config['chunk_ovlp'],
        root_path=downstream_path,
        gpt_only=not config['use_encoder'],
    )
    train_dataset = train_gpt.MotorImageryDataset(train_files, **dataset_kwargs)
    test_dataset = train_gpt.MotorImageryDataset(test_files, **dataset_kwargs)
    # Matches train_gpt.py's own decoding-mode branch exactly (including
    # this validation/test swap), for apples-to-apples comparison.
    validation_dataset = test_dataset
    test_dataset = train_dataset

    def model_init(params: Dict = None):
        model_config = dict(config)
        if params is not None:
            model_config |= params
        return make_prompt_tuned_model(model_config)

    trainer = train_gpt.make_trainer(
        model_init=model_init,
        training_style=config['training_style'],
        run_name=config['run_name'],
        output_dir=config['log_dir'],
        train_dataset=train_dataset,
        validation_dataset=validation_dataset,
        per_device_train_batch_size=config['per_device_training_batch_size'],
        per_device_eval_batch_size=config['per_device_validation_batch_size'],
        dataloader_num_workers=config['num_workers'],
        optim=config['optim'],
        learning_rate=config['learning_rate'],
        weight_decay=config['weight_decay'],
        adam_beta1=config['adam_beta_1'],
        adam_beta2=config['adam_beta_1'],
        adam_epsilon=config['adam_epsilon'],
        max_grad_norm=config['max_grad_norm'],
        lr_scheduler_type=config['lr_scheduler'],
        warmup_ratio=config['warmup_ratio'],
        max_steps=config['training_steps'],
        save_steps=config['training_steps'] * 2,
        logging_steps=config['log_every_n_steps'],
        eval_steps=config['eval_every_n_steps'],
        seed=config['seed'] if config['set_seed'] else np.random.choice(range(1, 100000)),
        fp16=config['fp16'],
        deepspeed=config['deepspeed'],
    )

    if config['do_train']:
        trainer.train(resume_from_checkpoint=config['resume_from'])
        trainer.save_model(os.path.join(config['log_dir'], 'model_final'))

    test_prediction = trainer.predict(test_dataset)
    pd.DataFrame(test_prediction.metrics, index=[0]).to_csv(
        os.path.join(config['log_dir'], 'test_metrics.csv'), index=False
    )
    np.save(os.path.join(config['log_dir'], 'test_predictions.npy'), test_prediction.predictions)
    np.save(os.path.join(config['log_dir'], 'test_label_ids.npy'), test_prediction.label_ids)

    return trainer


if __name__ == '__main__':
    trainer = train()

In [ ]:
%%writefile tests/test_soft_prompt.py
#!/usr/bin/env python3
"""
Smoke test for the class-conditional soft prompt-tuning novelty.

No real EEG data or the released pretrained checkpoint is available in
this environment, so this builds a tiny random GPT2Config decoder and
fabricated tensors matching the shapes NeuroGPT's own pipeline produces,
and checks that:
  - the soft-prompt menu prepends the right number of tokens,
  - forward + backward runs in decoding mode,
  - num_tokens_per_condition=0 degenerates to a no-op (frozen baseline),
  - freezing leaves only the prompt menu + pooler + decoding head trainable.

Run with: python tests/test_soft_prompt.py
"""
import os
import sys

import torch

_TEST_DIR = os.path.dirname(os.path.realpath(__file__))
_SRC_DIR = os.path.join(os.path.dirname(_TEST_DIR), 'src')
if _SRC_DIR not in sys.path:
    sys.path.insert(0, _SRC_DIR)

from decoder.gpt import GPTModel  # noqa: E402
from embedder.base import BaseEmbedder  # noqa: E402
from prompt_tuning.soft_prompt import ConditionSoftPrompt, PromptTunedModel, freeze_for_prompt_tuning  # noqa: E402

EMBED_DIM = 32
NUM_CONDITIONS = 4
NUM_TOKENS_PER_CONDITION = 3
BATCH_SIZE = 2
NUM_CHUNKS = 5


def _make_model(num_tokens_per_condition):
    embedder = BaseEmbedder(in_dim=EMBED_DIM, embed_dim=EMBED_DIM, num_hidden_layers=1)
    decoder = GPTModel(num_hidden_layers=2, num_attention_heads=4, embed_dim=EMBED_DIM, n_positions=64)
    model = PromptTunedModel(
        encoder=None, embedder=embedder, decoder=decoder, unembedder=None,
        num_conditions=NUM_CONDITIONS, num_tokens_per_condition=num_tokens_per_condition,
    )
    model.switch_decoding_mode(is_decoding_mode=True, num_decoding_classes=NUM_CONDITIONS)
    return model


def _make_batch():
    inputs = torch.randn(BATCH_SIZE, NUM_CHUNKS, EMBED_DIM)
    attention_mask = torch.ones(BATCH_SIZE, NUM_CHUNKS, dtype=torch.long)
    attention_mask[0, -1] = 0  # exercise padding / last-token indexing
    labels = torch.randint(0, NUM_CONDITIONS, (BATCH_SIZE,))
    return {'inputs': inputs, 'attention_mask': attention_mask, 'labels': labels}


def test_prompt_prepends_correct_shape():
    prompt = ConditionSoftPrompt(NUM_CONDITIONS, NUM_TOKENS_PER_CONDITION, EMBED_DIM)
    inputs_embeds = torch.randn(BATCH_SIZE, NUM_CHUNKS, EMBED_DIM)
    attention_mask = torch.ones(BATCH_SIZE, NUM_CHUNKS, dtype=torch.long)

    new_embeds, new_mask = prompt(inputs_embeds, attention_mask)

    expected_prompt_len = NUM_CONDITIONS * NUM_TOKENS_PER_CONDITION
    assert new_embeds.shape == (BATCH_SIZE, expected_prompt_len + NUM_CHUNKS, EMBED_DIM)
    assert new_mask.shape == (BATCH_SIZE, expected_prompt_len + NUM_CHUNKS)
    assert torch.all(new_mask[:, :expected_prompt_len] == 1)


def test_zero_tokens_is_a_noop():
    prompt = ConditionSoftPrompt(NUM_CONDITIONS, 0, EMBED_DIM)
    inputs_embeds = torch.randn(BATCH_SIZE, NUM_CHUNKS, EMBED_DIM)
    attention_mask = torch.ones(BATCH_SIZE, NUM_CHUNKS, dtype=torch.long)

    new_embeds, new_mask = prompt(inputs_embeds, attention_mask)

    assert torch.equal(new_embeds, inputs_embeds)
    assert torch.equal(new_mask, attention_mask)


def test_forward_and_backward_run():
    model = _make_model(NUM_TOKENS_PER_CONDITION)
    batch = _make_batch()

    losses, outputs = model.compute_loss(batch=batch, return_outputs=True)

    assert 'decoding_logits' in outputs
    assert outputs['decoding_logits'].shape == (BATCH_SIZE, NUM_CONDITIONS)

    losses['loss'].backward()
    assert model.soft_prompt.prompt_embeds.grad is not None


def test_zero_prompt_tokens_baseline_runs():
    model = _make_model(0)
    batch = _make_batch()

    losses, outputs = model.compute_loss(batch=batch, return_outputs=True)
    assert outputs['decoding_logits'].shape == (BATCH_SIZE, NUM_CONDITIONS)
    losses['loss'].backward()


def test_freezing_leaves_only_prompt_and_head_trainable():
    model = _make_model(NUM_TOKENS_PER_CONDITION)
    freeze_for_prompt_tuning(model)

    for n, p in model.named_parameters():
        is_trainable_group = n.startswith('soft_prompt.') or 'pooler_layer' in n or 'decoding_head' in n
        assert p.requires_grad == is_trainable_group, f'unexpected requires_grad for {n}: {p.requires_grad}'

    assert any(n.startswith('embedder.') and not p.requires_grad for n, p in model.named_parameters())


if __name__ == '__main__':
    test_prompt_prepends_correct_shape()
    test_zero_tokens_is_a_noop()
    test_forward_and_backward_run()
    test_zero_prompt_tokens_baseline_runs()
    test_freezing_leaves_only_prompt_and_head_trainable()
    print('All soft-prompt smoke tests passed.')

## 3. Smoke test
Verifies the mechanism (prepend shapes, forward+backward, freezing) with fabricated tensors — no data or checkpoint needed yet.

In [ ]:
!python tests/test_soft_prompt.py

## 2. Pretrained checkpoint

Downloaded from the vendor's own HuggingFace repo, linked from their README: https://huggingface.co/wenhuic/Neuro-GPT/tree/main

In [ ]:
!mkdir -p pretrained_model
!wget -q --show-progress -O pretrained_model/pytorch_model.bin \
  "https://huggingface.co/wenhuic/Neuro-GPT/resolve/main/pretrained_model/pytorch_model.bin"
!ls -lh pretrained_model/  # expect ~303MB (318189905 bytes) -- verified against the HF repo's own API listing

## 3. Prepare BCI Competition IV 2a

`src/batcher/downstream_dataset.py::MotorImageryDataset` expects, per subject/session, an `.npz` with keys `s`, `etyp`, `epos`, `edur`, `artifacts` (the original BioSig/GDF field names), plus a `true_labels/<name>.mat` file. The vendored repo does not ship a converter for this, so this notebook builds one from the raw `.gdf` files.

**Manual step (data is behind a usage agreement, so this can't be scripted):**
1. Go to https://www.bbci.de/competition/iv/#dataset2a, agree to the terms, and download the 18 raw `.gdf` files (`A01T.gdf` ... `A09E.gdf`) plus `true_labels.zip`.
2. Upload them to your Google Drive under `neurogpt_data/raw_bci2a/` (and the unzipped label files under `neurogpt_data/raw_bci2a/true_labels/`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

RAW_DIR = '/content/drive/MyDrive/neurogpt_data/raw_bci2a'
OUT_DIR = '/content/drive/MyDrive/neurogpt_data/bci2a_egg_npz'
RESULTS_DIR = '/content/drive/MyDrive/neurogpt_results'
import os
assert os.path.isdir(RAW_DIR), f'Upload the raw BCI-IV-2a files to {RAW_DIR} first (see markdown above).'

Conversion notes (verify the sanity-check counts below before trusting a full run):
- `s`: continuous signal in microvolts, shape `(n_times, n_channels)` — `MotorImageryDataset` transposes it back internally and slices the first 22 channels (EEG only; the GDF files also carry 3 EOG channels after the 22 EEG channels, in that order).
- `etyp`/`epos`/`edur`: GDF annotation `description` (the original numeric BioSig event code, e.g. `768`=trial start, `769..772`=class cues, `1023`=rejected) and `onset`/`duration` converted from seconds to samples via the file's own sampling rate.
- `artifacts`: not read by `MotorImageryDataset`'s trial-extraction code path, so this is filled with zeros rather than parsed.

In [ ]:
!pip install -q mne
import mne, numpy as np, os, shutil

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, 'true_labels'), exist_ok=True)

gdf_files = sorted(f for f in os.listdir(RAW_DIR) if f.endswith('.gdf'))
print(f'Found {len(gdf_files)} .gdf files (expect 18: A01T..A09T, A01E..A09E)')

for fname in gdf_files:
    subj_name = fname.replace('.gdf', '')
    raw = mne.io.read_raw_gdf(os.path.join(RAW_DIR, fname), preload=True, verbose=False)
    sfreq = raw.info['sfreq']
    data = raw.get_data()  # (n_channels, n_times), volts, original GDF channel order
    s = (data * 1e6).T     # -> microvolts, (n_times, n_channels)

    onsets_samp = np.round(np.array(raw.annotations.onset) * sfreq).astype(int)
    durations_samp = np.round(np.array(raw.annotations.duration) * sfreq).astype(int)
    etyp = np.array([int(d) for d in raw.annotations.description]).reshape(-1, 1)
    epos = onsets_samp.reshape(-1, 1)
    edur = durations_samp.reshape(-1, 1)
    artifacts = np.zeros((len(etyp), 1))

    np.savez(
        os.path.join(OUT_DIR, subj_name + '.npz'),
        s=s, etyp=etyp, epos=epos, edur=edur, artifacts=artifacts,
    )

    n_trial_starts = int((etyp == 768).sum())
    print(f'{subj_name}: {s.shape[0]} samples, {s.shape[1]} channels, {n_trial_starts} trial-start (768) events')

true_labels_src = os.path.join(RAW_DIR, 'true_labels')
for f in os.listdir(true_labels_src):
    shutil.copy(os.path.join(true_labels_src, f), os.path.join(OUT_DIR, 'true_labels', f))

print('Done. --dst-data-path =', OUT_DIR)

## 5. Run the novelty

Same fold, same core hyperparameters as the baseline notebook, so the two are directly comparable — the only real difference is the adaptation mechanism. Leave the original `--freeze-*` flags at their defaults (`False`); `--freeze-backbone` does the freezing instead, uniformly across encoder/embedder/decoder-transformer.

In [ ]:
!python src/prompt_tuning/train_prompt_tuning.py \
  --training-style='decoding' \
  --num-decoding-classes=4 \
  --training-steps=10000 \
  --eval_every_n_steps=500 \
  --log-every-n-steps=1000 \
  --num_chunks=2 \
  --per-device-training-batch-size=32 \
  --per-device-validation-batch-size=32 \
  --chunk_len=500 \
  --chunk_ovlp=0 \
  --run-name='novelty_prompt_tuning_colab' \
  --fold_i=0 \
  --num-encoder-layers=6 \
  --num-hidden-layers=6 \
  --learning-rate=1e-4 \
  --use-encoder='True' \
  --embedding-dim=1024 \
  --pretrained-model='pretrained_model/pytorch_model.bin' \
  --dst-data-path="{OUT_DIR}/" \
  --log-dir="{RESULTS_DIR}" \
  --num-prompt-tokens-per-condition=4 \
  --freeze-backbone='True'

## 6. Check results

In [ ]:
import pandas as pd, numpy as np, os
RUN_DIR = os.path.join(RESULTS_DIR, 'novelty_prompt_tuning_colab-0')  # run-name + '-' + fold_i, per train_gpt.py's get_config()
print(pd.read_csv(os.path.join(RUN_DIR, 'test_metrics.csv')))
preds = np.load(os.path.join(RUN_DIR, 'test_predictions.npy'))
labels = np.load(os.path.join(RUN_DIR, 'test_label_ids.npy'))
acc = (preds.argmax(-1) == labels).mean()
print('novelty_prompt_tuning_colab test accuracy:', acc)

## 7. Optional: frozen linear-probe ablation (free — no prompt tokens at all)
`--num-prompt-tokens-per-condition=0` degenerates the novelty to a plain frozen-backbone linear probe, using the exact same code path — a clean 3-way comparison point.

In [ ]:
!python src/prompt_tuning/train_prompt_tuning.py \
  --training-style='decoding' \
  --num-decoding-classes=4 \
  --training-steps=10000 \
  --eval_every_n_steps=500 \
  --log-every-n-steps=1000 \
  --num_chunks=2 \
  --per-device-training-batch-size=32 \
  --per-device-validation-batch-size=32 \
  --chunk_len=500 \
  --chunk_ovlp=0 \
  --run-name='linear_probe_ablation_colab' \
  --fold_i=0 \
  --num-encoder-layers=6 \
  --num-hidden-layers=6 \
  --learning-rate=1e-4 \
  --use-encoder='True' \
  --embedding-dim=1024 \
  --pretrained-model='pretrained_model/pytorch_model.bin' \
  --dst-data-path="{OUT_DIR}/" \
  --log-dir="{RESULTS_DIR}" \
  --num-prompt-tokens-per-condition=0 \
  --freeze-backbone='True'

## 8. Compare all three (run the baseline notebook first)
Reads results from the same shared `RESULTS_DIR` on Drive, so this works once both notebooks have been run at least once.

In [ ]:
import pandas as pd, numpy as np, os

def load_acc(run_name):
    run_dir = os.path.join(RESULTS_DIR, run_name + '-0')
    if not os.path.isdir(run_dir):
        return None
    preds = np.load(os.path.join(run_dir, 'test_predictions.npy'))
    labels = np.load(os.path.join(run_dir, 'test_label_ids.npy'))
    return (preds.argmax(-1) == labels).mean()

comparison = pd.DataFrame([
    {'run': 'actual_baseline (ft-only-encoder)', 'accuracy': load_acc('actual_baseline_colab')},
    {'run': 'novelty (soft prompt-tuning, 4 tokens/condition)', 'accuracy': load_acc('novelty_prompt_tuning_colab')},
    {'run': 'ablation (frozen linear probe, 0 prompt tokens)', 'accuracy': load_acc('linear_probe_ablation_colab')},
])
comparison